In [8]:
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, UnstructuredMarkdownLoader, UnstructuredWordDocumentLoader, UnstructuredPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json
import re


# 加载数据集
def load_data() -> list[Document]:
    # 加载文本数据
    text_documents : list[Document] = TextLoader(file_path="./knowledge_base/sample.txt" ,encoding="utf-8").load()
    # 加载 md
    md_documents: list [Document] = UnstructuredMarkdownLoader(file_path="./knowledge_base/sample.md",mode="single", strategy= "hi_res").load()
    # 加载 docx
    docx_documents : list[Document] = UnstructuredWordDocumentLoader(file_path="./knowledge_base/sample.docx",mode="single", strategy = "hi_res").load()
    # 加载 PDF
    pdf_documents :list[Document] = UnstructuredPDFLoader(file_path="./knowledge_base/sample.pdf",mode="elements", strategy = "hi_res",infer_table_structure = True, languages=["eng", "chi_sim"] ).load()
    return text_documents  + md_documents+  docx_documents +pdf_documents

# 清洗数据
def clean_documents(documents: list[Document]) -> list[Document]:
    result: list[Document] = []
    for document in documents:
        # 清洗内容
        content = document.page_content
        content = content.strip()
        content = re.sub(r"\n{2,}","\n",content)
        document.page_content = content
        
        # chroma 支持的类型
        chroma_db_support_type = (int , float, str , bool)
        for key, value in document.metadata.items():
            if not isinstance(value, chroma_db_support_type):
                try:
                    document.metadata[key] = json.dumps(value, ensure_ascii=False)
                except:
                    print(key,"序列化失败")
                    document.metadata[key] = str(value)

        result.append(document)
    return result

# 切片
def split_documents(documents: list[Document]) -> list[Document]:
    result: list[Document] = []
    text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n"], chunk_size=500,chunk_overlap=200)
    result = text_splitter.split_documents(documents)
    return result

# 构建索引
# 加载到 ChromaDB 中
def getEmbdding()-> HuggingFaceEmbeddings:
    model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-base-zh-v1.5",
        model_kwargs = {"device": "cpu"},
        encode_kwargs = {"normalize_embeddings": False}
    )
    return model

def save_chromadb(documents: list[Document] ,model: Embeddings ):
     Chroma.from_documents(documents = documents, embedding= model,persist_directory = "./vectorstore")

In [3]:
document:list[Document] = load_data()

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 12660.88it/s]


In [5]:
document

[Document(metadata={'source': './knowledge_base/sample.txt'}, page_content='始计第一\n孙子曰：兵者，国之大事，死生之地，存亡之道，不可不察也。\n故经之以五事，校之以计，而索其情：一曰道，二曰天，三曰地，四曰将，五曰法。\n道者，令民与上同意，可与之死，可与之生，而不畏危也；天者，阴阳、寒暑、时制也；地者，远近、险易、广狭、死生也；将者，智、信、仁、勇、严也；法者，曲制、官道、主用也。凡此五者，将莫不闻，知之者胜，不知者不胜。\n故校之以计，而索其情，曰：主孰有道？将孰有能？天地孰得？法令孰行？兵众孰强？士卒孰练？赏罚孰明？吾以此知胜负矣。\n将听吾计，用之必胜，留之；将不听吾计，用之必败，去之。\n计利以听，乃为之势，以佐其外。势者，因利而制权也。\n兵者，诡道也。故能而示之不能，用而示之不用，近而示之远，远而示之近。利而诱之，乱而取之，实而备之，强而避之，怒而挠之，卑而骄之，佚而劳之，亲而离之，攻其无备，出其不意。此兵家之胜，不可先传也。\n夫未战而庙算胜者，得算多也；未战而庙算不胜者，得算少也。多算胜，少算不胜，而况无算乎！吾以此观之，胜负见矣。\n作战第二\n孙子曰：凡用兵之法，驰车千驷，革车千乘，带甲十万，千里馈粮。则内外之费，宾客之用，胶漆之材，车甲之奉，日费千金，然后十万之师举矣。\n其用战也，贵胜，久则钝兵挫锐，攻城则力屈，久暴师则国用不足。夫钝兵挫锐，屈力殚货，则诸侯乘其弊而起，虽有智者，不能善其后矣。故兵闻拙速，未睹巧之久也。夫兵久而国利者，未之有也。故不尽知用兵之害者，则不能尽知用兵之利也。\n善用兵者，役不再籍，粮不三载，取用于国，因粮于敌，故军食可足也。国之贫于师者远输，远输则百姓贫；近于师者贵卖，贵卖则百姓竭，财竭则急于丘役。力屈财殚，中原内虚于家，百姓之费，十去其七；公家之费，破军罢马，甲胄矢弩，戟楯矛橹，丘牛大车，十去其六。\n故智将务食于敌，食敌一钟，当吾二十钟；萁秆一石，当吾二十石。故杀敌者，怒也；取敌之利者，货也。故车战，得车十乘以上，赏其先得者，而更其旌旗。车杂而乘之，卒善而养之，是谓胜敌而益强。\n故兵贵胜，不贵久。故知兵之将，民之司命。国家安危之主也。\n谋攻第三\n孙子曰：凡用兵之法，全国为上，破国次之；全军为上，破军次之；全旅为上，破旅次之；

In [4]:
clean_data : list[Document] = clean_documents(document)
print(clean_data)

[Document(metadata={'source': './knowledge_base/sample.txt'}, page_content='始计第一\n孙子曰：兵者，国之大事，死生之地，存亡之道，不可不察也。\n故经之以五事，校之以计，而索其情：一曰道，二曰天，三曰地，四曰将，五曰法。\n道者，令民与上同意，可与之死，可与之生，而不畏危也；天者，阴阳、寒暑、时制也；地者，远近、险易、广狭、死生也；将者，智、信、仁、勇、严也；法者，曲制、官道、主用也。凡此五者，将莫不闻，知之者胜，不知者不胜。\n故校之以计，而索其情，曰：主孰有道？将孰有能？天地孰得？法令孰行？兵众孰强？士卒孰练？赏罚孰明？吾以此知胜负矣。\n将听吾计，用之必胜，留之；将不听吾计，用之必败，去之。\n计利以听，乃为之势，以佐其外。势者，因利而制权也。\n兵者，诡道也。故能而示之不能，用而示之不用，近而示之远，远而示之近。利而诱之，乱而取之，实而备之，强而避之，怒而挠之，卑而骄之，佚而劳之，亲而离之，攻其无备，出其不意。此兵家之胜，不可先传也。\n夫未战而庙算胜者，得算多也；未战而庙算不胜者，得算少也。多算胜，少算不胜，而况无算乎！吾以此观之，胜负见矣。\n作战第二\n孙子曰：凡用兵之法，驰车千驷，革车千乘，带甲十万，千里馈粮。则内外之费，宾客之用，胶漆之材，车甲之奉，日费千金，然后十万之师举矣。\n其用战也，贵胜，久则钝兵挫锐，攻城则力屈，久暴师则国用不足。夫钝兵挫锐，屈力殚货，则诸侯乘其弊而起，虽有智者，不能善其后矣。故兵闻拙速，未睹巧之久也。夫兵久而国利者，未之有也。故不尽知用兵之害者，则不能尽知用兵之利也。\n善用兵者，役不再籍，粮不三载，取用于国，因粮于敌，故军食可足也。国之贫于师者远输，远输则百姓贫；近于师者贵卖，贵卖则百姓竭，财竭则急于丘役。力屈财殚，中原内虚于家，百姓之费，十去其七；公家之费，破军罢马，甲胄矢弩，戟楯矛橹，丘牛大车，十去其六。\n故智将务食于敌，食敌一钟，当吾二十钟；萁秆一石，当吾二十石。故杀敌者，怒也；取敌之利者，货也。故车战，得车十乘以上，赏其先得者，而更其旌旗。车杂而乘之，卒善而养之，是谓胜敌而益强。\n故兵贵胜，不贵久。故知兵之将，民之司命。国家安危之主也。\n谋攻第三\n孙子曰：凡用兵之法，全国为上，破国次之；全军为上，破军次之；全旅为上，破旅次之；

In [9]:
getEmbdding()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 59263.45it/s]
BertModel LOAD REPORT from: BAAI/bge-base-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFaceEmbeddings(model_name='BAAI/bge-base-zh-v1.5', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': False}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [10]:
save_chromadb(clean_data,getEmbdding())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 48833.75it/s]
BertModel LOAD REPORT from: BAAI/bge-base-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
